# 18 多层感知机 MLP Neural Network

依赖安装说明：`pip install numpy matplotlib scikit-learn torch`

MLP 是最基础的前馈神经网络。它通过多层线性变换和非线性激活函数，学习非线性决策边界。


## 0. 学习目标和阅读地图

MLP 是深度学习最基础的网络。你需要掌握：

1. 为什么线性层之间必须加非线性激活。
2. 前向传播和反向传播分别在做什么。
3. 隐藏层宽度、深度如何影响表达能力。
4. PyTorch 里的 `zero_grad -> backward -> step` 为什么是固定流程。


## 1. 数学逻辑

一层神经网络通常是：

$$h = \phi(W_1x+b_1)$$

输出层：

$$\hat y = W_2h+b_2$$

如果没有非线性激活 `phi`，多层线性变换仍然等价于一层线性变换。常见激活函数 ReLU：

$$ReLU(x)=\max(0,x)$$

二分类常用交叉熵，训练依赖反向传播计算梯度。


## 1.1 推导拆开看：链式法则

MLP 的一层可以写成：

$$h=\phi(Wx+b)$$

loss 对参数的梯度通过链式法则传回来：

$$\frac{\partial L}{\partial W}=\frac{\partial L}{\partial h}\frac{\partial h}{\partial z}\frac{\partial z}{\partial W}$$

其中 `z = Wx + b`。反向传播并不是魔法，它只是高效地重复使用链式法则。

ReLU 的导数很简单：

$$ReLU'(z)=1 \text{ if } z>0, \text{ else } 0$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_moons(n_samples=400, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 1.2 MLP 的数据流

本例是二维输入、二分类输出：

- 输入层：2 个数。
- 隐藏层：把 2 维映射到更高维空间。
- ReLU：引入非线性。
- 输出层：输出 2 个 logits，对应两个类别。

分类时不是直接对 logits 当概率，而是交给 `CrossEntropyLoss`，它内部会做 log-softmax。


In [ ]:
# 从零实现：一个很小的 numpy MLP，结构 2 -> 8 -> 1
rng = np.random.default_rng(42)
W1 = rng.normal(scale=0.5, size=(2, 8))
b1 = np.zeros(8)
W2 = rng.normal(scale=0.5, size=(8, 1))
b2 = np.zeros(1)
lr = 0.1

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

for step in range(800):
    z1 = X_train_s @ W1 + b1
    h = relu(z1)
    logits = h @ W2 + b2
    p = sigmoid(logits[:, 0])
    loss = -np.mean(y_train * np.log(p + 1e-12) + (1-y_train) * np.log(1-p + 1e-12))

    dlogits = (p - y_train)[:, None] / len(X_train_s)
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(axis=0)
    dh = dlogits @ W2.T
    dz1 = dh * (z1 > 0)
    dW1 = X_train_s.T @ dz1
    db1 = dz1.sum(axis=0)

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

print('最后 loss:', round(loss, 3))
test_p = sigmoid((relu(X_test_s @ W1 + b1) @ W2 + b2)[:, 0])
print('从零 MLP accuracy:', round(accuracy_score(y_test, test_p >= 0.5), 3))


## 1.3 从零实现代码怎么读

NumPy 版本完整展示了一次手写反向传播：

1. 前向：`z1 -> h -> logits -> p -> loss`。
2. 反向：从 `dlogits` 开始逐层算回 `dW2/db2/dW1/db1`。
3. 更新：每个参数减去学习率乘梯度。

如果你能读懂这一段，PyTorch 的 `loss.backward()` 就不再神秘。


In [ ]:
# PyTorch 实战
import torch
from torch import nn

Xtr = torch.tensor(X_train_s, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test_s, dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.CrossEntropyLoss()

for step in range(300):
    logits = model(Xtr)
    loss = loss_fn(logits, ytr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(Xte).argmax(dim=1).numpy()
print('PyTorch MLP accuracy:', round(accuracy_score(y_test, pred), 3))


In [ ]:
# 诊断：画 PyTorch MLP 的决策边界
xx, yy = np.meshgrid(np.linspace(X_train_s[:,0].min()-1, X_train_s[:,0].max()+1, 180),
                     np.linspace(X_train_s[:,1].min()-1, X_train_s[:,1].max()+1, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
with torch.no_grad():
    zz = model(torch.tensor(grid, dtype=torch.float32)).argmax(dim=1).numpy().reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train_s[:,0], X_train_s[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=24)
plt.title('MLP 学到的非线性决策边界')
plt.show()


## 2.1 如何诊断 MLP

深度学习模型要同时看训练集和验证集。常见信号：

- 训练 loss 降、验证 loss 升：过拟合。
- 两者都高：欠拟合或学习率不合适。
- loss 震荡：学习率可能太大。
- 准确率卡住：模型太小、特征不足或优化困难。


## 2. 常见误区

- 没有非线性激活，多层网络仍然只是线性模型。
- 网络越大越容易过拟合，需要验证集、正则化和早停。
- 学习率太大 loss 会震荡，太小训练很慢。

## 3. 小实验

- 改隐藏层宽度 `16`。
- 把 ReLU 换成 Tanh。
- 增加噪声，看模型过拟合边界。


## 5. 复习清单

- MLP = 线性层 + 非线性激活的堆叠。
- 没有非线性，多层线性仍等价于一层线性。
- PyTorch 的训练步骤是清梯度、反传、更新。
- 模型容量越大，越需要验证集和正则化。
